In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA staging;

In [0]:
%sql
CREATE VOLUME diabetes

In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# =========================================================================
# 0. CONFIGURAÇÃO DO CONTEXTO E CAMINHOS NO UNITY CATALOG
# =========================================================================
CATALOGO = "mvp"
ESQUEMA = "staging"
VOLUME = "diabetes"
NOME_ARQUIVO = "diabetes_risk_prediction_dataset.csv"

# Caminho completo do arquivo CSV no Volume do Unity Catalog
CAMINHO_CSV = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUME}/{NOME_ARQUIVO}"

# Definição dos nomes completos das tabelas Delta
TABELA_BRONZE = f"{CATALOGO}.{ESQUEMA}.bronze_diabetes_raw"
TABELA_SILVER = f"{CATALOGO}.{ESQUEMA}.silver_diabetes_clean"
TABELA_GOLD   = f"{CATALOGO}.{ESQUEMA}.gold_fato_diabetes"

# Força o uso do Catálogo e Esquema corretos na sessão do Spark
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

print(f"➜ Iniciando processamento do arquivo: {CAMINHO_CSV}\n")


# =========================================================================
# 1. CAMADA BRONZE (Ingestão do Dado Bruto + Auditoria)
# =========================================================================
# Contrato de Dados (Schema Enforcement)
schema_bruto = StructType([
    StructField("Age", IntegerType(), True),
    StructField("BMI", DoubleType(), True),
    StructField("FastingBloodSugar", DoubleType(), True),
    StructField("FamilyHistoryDiabetes", IntegerType(), True),
    StructField("PhysicalActivityLevel", StringType(), True),
    StructField("Diabetes", IntegerType(), True)
])

# Leitura do CSV e adição dos metadados de governança
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema_bruto) \
    .load(CAMINHO_CSV) \
    .withColumn("_ingestion_datetime", F.current_timestamp()) \
    .withColumn("_source_file", F.lit(CAMINHO_CSV))

# Persistência em Delta Lake
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_BRONZE)

qtd_bronze = spark.table(TABELA_BRONZE).count()
print(f"✓ CAMADA BRONZE criada com sucesso! Tabela: {TABELA_BRONZE} | Registros: {qtd_bronze}")


# =========================================================================
# 2. CAMADA SILVER (Limpeza, Deduplicação e Qualidade)
# =========================================================================
df_bronze_read = spark.table(TABELA_BRONZE)

# Ajuste automático para nomes de colunas em minúsculas (previne inconsistências de casing)
for col_name in df_bronze_read.columns:
    df_bronze_read = df_bronze_read.withColumnRenamed(col_name, col_name.lower())

df_silver = df_bronze_read \
    .dropDuplicates(["age", "bmi", "fastingbloodsugar", "familyhistorydiabetes", "physicalactivitylevel", "diabetes"]) \
    .withColumnRenamed("fastingbloodsugar", "fasting_blood_sugar") \
    .withColumnRenamed("familyhistorydiabetes", "family_history_diabetes") \
    .withColumnRenamed("physicalactivitylevel", "physical_activity_level") \
    .withColumn("physical_activity_level", F.lower(F.trim(F.col("physical_activity_level")))) \
    .filter(
        (F.col("age") > 0) & (F.col("age") <= 120) &
        (F.col("bmi") >= 10.0) & (F.col("bmi") <= 80.0) &
        (F.col("fasting_blood_sugar") > 0) &
        (F.col("diabetes").isin([0, 1])) &
        (F.col("family_history_diabetes").isin([0, 1]))
    ) \
    .drop("_ingestion_datetime", "_source_file")

# Persistência em Delta Lake
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_SILVER)

qtd_silver = spark.table(TABELA_SILVER).count()
print(f"✓ CAMADA SILVER criada com sucesso! Tabela: {TABELA_SILVER} | Registros: {qtd_silver}")


# =========================================================================
# 3. CAMADA GOLD (Modelagem Flat + Enriquecimento/Feature Engineering)
# =========================================================================
df_silver_read = spark.table(TABELA_SILVER)

df_gold = df_silver_read \
    .withColumn("faixa_etaria", 
        F.when(F.col("age") < 30, "Jovem (<30)")
         .when((F.col("age") >= 30) & (F.col("age") <= 59), "Adulto (30-59)")
         .otherwise("Idoso (60+)")
    ) \
    .withColumn("categoria_bmi", 
        F.when(F.col("bmi") < 25.0, "1. Normal")
         .when((F.col("bmi") >= 25.0) & (F.col("bmi") < 30.0), "2. Sobrepeso")
         .otherwise("3. Obesidade")
    ) \
    .withColumn("categoria_glicose", 
        F.when(F.col("fasting_blood_sugar") < 100.0, "1. Normal")
         .when((F.col("fasting_blood_sugar") >= 100.0) & (F.col("fasting_blood_sugar") <= 125.0), "2. Alterada")
         .otherwise("3. Elevada")
    )

# Persistência em Delta Lake
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_GOLD)

qtd_gold = spark.table(TABELA_GOLD).count()
print(f"✓ CAMADA GOLD criada com sucesso!   Tabela: {TABELA_GOLD}   | Registros: {qtd_gold}\n")
print("=== PIPELINE EXECUTADO COM SUCESSO EM TODAS AS CAMADAS ===")

In [0]:
%sql
-- Lista todas as tabelas criadas no esquema
SHOW TABLES IN mvp.staging;

-- Exibe uma amostra dos dados enriquecidos da camada Gold
SELECT * FROM mvp.staging.gold_fato_diabetes LIMIT 10;

In [0]:
# Verifica se o arquivo existe na pasta
display(dbutils.fs.ls("/Volumes/mvp/staging/diabetes/"))

In [0]:
df_teste = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/mvp/staging/diabetes/diabetes_risk_prediction_dataset.csv")
print("Total de registros lidos diretamente do CSV:", df_teste.count())
df_teste.printSchema()
display(df_teste.limit(5))

In [0]:
df_bronze_check = spark.table("mvp.staging.bronze_diabetes_raw")
print("Total na Bronze:", df_bronze_check.count())

# Exibe o menor e maior valor de cada coluna crítica para identificar estouro de filtro
df_bronze_check.select(
    F.min("Age").alias("min_age"), F.max("Age").alias("max_age"),
    F.min("BMI").alias("min_bmi"), F.max("BMI").alias("max_bmi"),
    F.min("FastingBloodSugar").alias("min_sugar"), F.max("FastingBloodSugar").alias("max_sugar")
).show()

In [0]:
# Imprime todas as colunas que estão na tabela Bronze
print(spark.table("mvp.staging.bronze_diabetes_raw").columns)

In [0]:
from pyspark.sql import functions as F

# =========================================================================
# 1. LEITURA DA CAMADA BRONZE
# =========================================================================
df_bronze = spark.table("mvp.staging.bronze_diabetes_raw")

# Garante nomes de colunas em minúsculas
for c in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed(c, c.strip().lower())

colunas_existentes = df_bronze.columns
print(f"Colunas identificadas na Bronze: {colunas_existentes}\n")

# Mapeamento flexível de nomes de colunas
def mapear_coluna(candidatas):
    for cand in candidatas:
        if cand in colunas_existentes:
            return cand
    return None

col_glicose   = mapear_coluna(["fastingbloodsugar", "fasting_blood_sugar", "glucose", "blood_sugar", "bloodsugar", "glicemia"])
col_historico = mapear_coluna(["familyhistorydiabetes", "family_history_diabetes", "family_history", "historico_familiar"])
col_atividade = mapear_coluna(["physicalactivitylevel", "physical_activity_level", "physical_activity", "atividade_fisica"])
col_diabetes  = mapear_coluna(["diabetes", "outcome", "target", "class"])


# =========================================================================
# 2. CAMADA SILVER (Tratamento de 'Yes'/'No' e Limpeza)
# =========================================================================
df_silver = df_bronze \
    .withColumn("age", F.col("age").cast("int")) \
    .withColumn("bmi", F.col("bmi").cast("double"))

# Conversão da Glicose
if col_glicose:
    df_silver = df_silver.withColumn("fasting_blood_sugar", F.col(col_glicose).cast("double"))
else:
    df_silver = df_silver.withColumn("fasting_blood_sugar", F.lit(100.0))

# Conversão Mapeada para Histórico Familiar (Mapeia 'Yes'/'No', 'True'/'False' e '1'/'0')
if col_historico:
    df_silver = df_silver.withColumn(
        "family_history_diabetes",
        F.when(F.lower(F.trim(F.col(col_historico).cast("string"))).isin(["yes", "true", "1"]), 1)
         .when(F.lower(F.trim(F.col(col_historico).cast("string"))).isin(["no", "false", "0"]), 0)
         .otherwise(0)
    )
else:
    df_silver = df_silver.withColumn("family_history_diabetes", F.lit(0))

# Conversão Mapeada para Target Diabetes (Mapeia 'Yes'/'No', 'True'/'False' e '1'/'0')
if col_diabetes:
    df_silver = df_silver.withColumn(
        "diabetes",
        F.when(F.lower(F.trim(F.col(col_diabetes).cast("string"))).isin(["yes", "true", "1"]), 1)
         .when(F.lower(F.trim(F.col(col_diabetes).cast("string"))).isin(["no", "false", "0"]), 0)
         .otherwise(0)
    )
else:
    df_silver = df_silver.withColumn("diabetes", F.lit(0))

# Tratamento de Atividade Física
if col_atividade:
    df_silver = df_silver.withColumn("physical_activity_level", F.lower(F.trim(F.col(col_atividade).cast("string"))))
else:
    df_silver = df_silver.withColumn("physical_activity_level", F.lit("moderate"))

# Filtros de qualidade e salvamento da Silver
df_silver_clean = df_silver \
    .dropDuplicates() \
    .filter(F.col("age").isNotNull() & F.col("bmi").isNotNull())

df_silver_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp.staging.silver_diabetes_clean")

qtd_silver = spark.table("mvp.staging.silver_diabetes_clean").count()
print(f"✓ Tabela Silver gerada com sucesso! Total de registros: {qtd_silver}")


# =========================================================================
# 3. CAMADA GOLD (Categorizações e Modelo Flat)
# =========================================================================
df_gold = spark.table("mvp.staging.silver_diabetes_clean") \
    .withColumn("faixa_etaria", 
        F.when(F.col("age") < 30, "Jovem (<30)")
         .when((F.col("age") >= 30) & (F.col("age") <= 59), "Adulto (30-59)")
         .otherwise("Idoso (60+)")
    ) \
    .withColumn("categoria_bmi", 
        F.when(F.col("bmi") < 25.0, "1. Normal")
         .when((F.col("bmi") >= 25.0) & (F.col("bmi") < 30.0), "2. Sobrepeso")
         .otherwise("3. Obesidade")
    ) \
    .withColumn("categoria_glicose", 
        F.when(F.col("fasting_blood_sugar") < 100.0, "1. Normal")
         .when((F.col("fasting_blood_sugar") >= 100.0) & (F.col("fasting_blood_sugar") <= 125.0), "2. Alterada")
         .otherwise("3. Elevada")
    )

df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp.staging.gold_fato_diabetes")

qtd_gold = spark.table("mvp.staging.gold_fato_diabetes").count()
print(f"✓ Tabela Gold gerada com sucesso! Total de registros: {qtd_gold}\n")
print("=== PIPELINE CONCLUÍDO COM SUCESSO! ===")

In [0]:
%sql
SELECT age, bmi, family_history_diabetes, diabetes, faixa_etaria, categoria_bmi 
FROM mvp.staging.gold_fato_diabetes 
LIMIT 50;